# Interactive VLM Inference Notebook

This notebook provides an interactive interface for testing your trained Vision Language Model (VLM). You can:

- Load your trained model
- Upload or select images
- Input custom prompts
- Generate responses in real-time
- Experiment with different generation parameters

## Requirements

Make sure you have the following packages installed:
```bash
pip install torch transformers pillow ipywidgets matplotlib
```

In [1]:
# Import Required Libraries
import torch
import os
from PIL import Image
import matplotlib.pyplot as plt
from transformers import CLIPProcessor
import ipywidgets as widgets
from IPython.display import display, clear_output
import io
import base64

# Import our custom model
from model import VisionLanguageModel

print("All libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

All libraries imported successfully!
PyTorch version: 2.7.1+cu128
CUDA available: True


## Load Pre-trained Model

Configure your model paths and load the trained VLM model.

In [2]:
# Model Configuration
MODEL_DIR = "checkpoints/instruction_qwen3_0.6b/final_model"  # Update this path to your trained model
VISION_ENCODER_NAME = "openai/clip-vit-large-patch14"

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load the model
try:
    print("Loading VLM model...")
    model = VisionLanguageModel.from_pretrained(MODEL_DIR)
    model = model.to(device)
    model.eval()
    
    # Get tokenizer from the model
    tokenizer = model.tokenizer
    
    # Load processor
    processor = CLIPProcessor.from_pretrained(VISION_ENCODER_NAME)
    
    print("✅ Model loaded successfully!")
    print(f"Model type: {type(model)}")
    print(f"Tokenizer vocab size: {len(tokenizer)}")
    
except Exception as e:
    print(f"❌ Error loading model: {e}")
    print("Please check the MODEL_DIR path and ensure the model exists.")

Using device: cuda
Loading VLM model...
Loading LLM from checkpoints/instruction_qwen3_0.6b/final_model/llm for modified weights.


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


✅ Model loaded successfully!
Model type: <class 'model.VisionLanguageModel'>
Tokenizer vocab size: 151670


## Helper Functions

Define utility functions for image processing and model inference.

In [3]:
def process_image(image_path_or_pil):
    """
    Process an image for the VLM model.
    
    Args:
        image_path_or_pil: Either a file path (str) or PIL Image object
    
    Returns:
        tuple: (processed_image_tensor, pil_image_for_display)
    """
    if isinstance(image_path_or_pil, str):
        image = Image.open(image_path_or_pil).convert('RGB')
    else:
        image = image_path_or_pil.convert('RGB')
    
    # Process image for the model
    processed_image = processor(images=image, return_tensors="pt")
    pixel_values = processed_image.pixel_values.to(device)
    
    return pixel_values, image

def generate_response(image_tensor, prompt, max_new_tokens=256, temperature=0.7, top_p=0.9):
    """
    Generate a response from the VLM model.
    
    Args:
        image_tensor: Processed image tensor
        prompt: Text prompt
        max_new_tokens: Maximum number of tokens to generate
        temperature: Sampling temperature
        top_p: Top-p sampling parameter
    
    Returns:
        str: Generated response
    """
    try:
        # Prepare the input prompt
        full_prompt = f"<image> USER: {prompt} ASSISTANT:"
        
        # Tokenize the prompt
        inputs = tokenizer(full_prompt, return_tensors="pt")
        input_ids = inputs.input_ids.to(device)
        attention_mask = inputs.attention_mask.to(device)
        
        # Generate response
        with torch.no_grad():
            outputs = model.generate(
                pixel_values=image_tensor,
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=temperature,
                top_p=top_p,
                pad_token_id=tokenizer.eos_token_id
            )
        
        # Decode the generated text
        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        # Extract only the assistant's response
        assistant_response = generated_text.split("ASSISTANT:")[-1].strip()
        
        return assistant_response
        
    except Exception as e:
        return f"Error generating response: {str(e)}"

def display_image_and_response(image, prompt, response):
    """Display image and response side by side."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # Display image
    ax1.imshow(image)
    ax1.axis('off')
    ax1.set_title("Input Image", fontsize=14, fontweight='bold')
    
    # Display text
    ax2.text(0.05, 0.95, f"Prompt: {prompt}", fontsize=12, fontweight='bold', 
             transform=ax2.transAxes, verticalalignment='top', wrap=True)
    ax2.text(0.05, 0.85, f"Response:\n{response}", fontsize=11, 
             transform=ax2.transAxes, verticalalignment='top', wrap=True)
    ax2.axis('off')
    ax2.set_title("Model Response", fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.show()

print("Helper functions defined successfully!")

Helper functions defined successfully!


## Prepare Sample Data

You can test with sample images or upload your own images.

In [4]:
# Sample prompts for testing
SAMPLE_PROMPTS = [
    "What do you see in this image?",
    "Describe the main objects in this image.",
    "What is happening in this scene?",
    "What colors are prominent in this image?",
    "Can you identify any people or animals in this image?",
    "What is the mood or atmosphere of this image?",
    "Are there any text or signs visible in this image?",
    "What time of day does this appear to be taken?",
    "What type of location or setting is this?",
    "What activities are taking place in this image?"
]

# Check for sample images in the data directory
def find_sample_images():
    """Find sample images in common directories."""
    possible_dirs = ["./data/image/images", "./data/images", "./images", "./samples"]
    sample_images = []
    
    for dir_path in possible_dirs:
        if os.path.exists(dir_path):
            for file in os.listdir(dir_path)[:5]:  # Get first 5 images
                if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif')):
                    sample_images.append(os.path.join(dir_path, file))
    
    return sample_images

# Find available sample images
sample_images = find_sample_images()
if sample_images:
    print(f"Found {len(sample_images)} sample images:")
    for img in sample_images:
        print(f"  - {img}")
else:
    print("No sample images found. You can upload your own images using the interface below.")

print(f"\nAvailable sample prompts: {len(SAMPLE_PROMPTS)}")

Found 5 sample images:
  - ./data/image/images/GCC_train_000000000.jpg
  - ./data/image/images/GCC_train_000000001.jpg
  - ./data/image/images/GCC_train_000000015.jpg
  - ./data/image/images/GCC_train_000000019.jpg
  - ./data/image/images/GCC_train_000000023.jpg

Available sample prompts: 10


## Interactive Inference Widget

Use the interactive controls below to test your VLM model with different images and prompts.

In [5]:
# Create interactive widgets
def create_inference_interface():
    """Create the main interactive inference interface."""
    
    # File upload widget
    file_upload = widgets.FileUpload(
        accept='image/*',
        multiple=False,
        description='Upload Image'
    )
    
    # Sample image dropdown
    sample_options = ['None'] + [os.path.basename(img) for img in sample_images]
    sample_dropdown = widgets.Dropdown(
        options=sample_options,
        value='None',
        description='Sample Image:'
    )
    
    # Prompt widgets
    prompt_dropdown = widgets.Dropdown(
        options=['Custom'] + SAMPLE_PROMPTS,
        value='Custom',
        description='Prompt:'
    )
    
    prompt_text = widgets.Textarea(
        value='What do you see in this image?',
        placeholder='Enter your custom prompt here...',
        description='Custom Prompt:',
        layout=widgets.Layout(width='400px', height='80px')
    )
    
    # Generation parameters
    max_tokens_slider = widgets.IntSlider(
        value=256,
        min=50,
        max=512,
        step=10,
        description='Max Tokens:'
    )
    
    temperature_slider = widgets.FloatSlider(
        value=0.7,
        min=0.1,
        max=1.0,
        step=0.1,
        description='Temperature:'
    )
    
    top_p_slider = widgets.FloatSlider(
        value=0.9,
        min=0.1,
        max=1.0,
        step=0.05,
        description='Top-p:'
    )
    
    # Generate button
    generate_button = widgets.Button(
        description='Generate Response',
        button_style='primary',
        icon='play'
    )
    
    # Output area
    output_area = widgets.Output()
    
    # Current image storage
    current_image = {'pil': None, 'tensor': None}
    
    def on_sample_change(change):
        """Handle sample image selection."""
        if change['new'] != 'None':
            img_path = None
            for img in sample_images:
                if os.path.basename(img) == change['new']:
                    img_path = img
                    break
            
            if img_path:
                try:
                    pixel_values, pil_image = process_image(img_path)
                    current_image['pil'] = pil_image
                    current_image['tensor'] = pixel_values
                    
                    with output_area:
                        clear_output()
                        print(f"✅ Loaded sample image: {change['new']}")
                        plt.figure(figsize=(8, 6))
                        plt.imshow(pil_image)
                        plt.axis('off')
                        plt.title(f"Selected Image: {change['new']}")
                        plt.show()
                except Exception as e:
                    with output_area:
                        clear_output()
                        print(f"❌ Error loading image: {e}")
    
    def on_upload_change(change):
        """Handle file upload."""
        if file_upload.value:
            try:
                # Get the uploaded file - handle both tuple and dict cases
                uploaded_files = file_upload.value
                if isinstance(uploaded_files, tuple):
                    # If it's a tuple, get the first element
                    uploaded_file = uploaded_files[0]
                elif isinstance(uploaded_files, dict):
                    # If it's a dict, get the first value
                    uploaded_file = list(uploaded_files.values())[0]
                else:
                    # Direct access if it's already the file object
                    uploaded_file = uploaded_files
                
                # Handle different file object structures
                if hasattr(uploaded_file, 'content'):
                    content = uploaded_file.content
                elif isinstance(uploaded_file, dict) and 'content' in uploaded_file:
                    content = uploaded_file['content']
                else:
                    raise ValueError("Unable to extract file content from uploaded file")
                
                image = Image.open(io.BytesIO(content))
                
                pixel_values, pil_image = process_image(image)
                current_image['pil'] = pil_image
                current_image['tensor'] = pixel_values
                
                # Reset sample dropdown
                sample_dropdown.value = 'None'
                
                with output_area:
                    clear_output()
                    print("✅ Image uploaded successfully!")
                    plt.figure(figsize=(8, 6))
                    plt.imshow(pil_image)
                    plt.axis('off')
                    plt.title("Uploaded Image")
                    plt.show()
            except Exception as e:
                with output_area:
                    clear_output()
                    print(f"❌ Error uploading image: {e}")
                    print(f"Debug info - file_upload.value type: {type(file_upload.value)}")
                    if hasattr(file_upload.value, '__dict__'):
                        print(f"Available attributes: {list(file_upload.value.__dict__.keys())}")
    
    def on_prompt_change(change):
        """Handle prompt dropdown change."""
        if change['new'] != 'Custom':
            prompt_text.value = change['new']
    
    def on_generate_click(b):
        """Handle generate button click."""
        if current_image['pil'] is None:
            with output_area:
                clear_output()
                print("❌ Please upload or select an image first!")
            return
        
        prompt = prompt_text.value.strip()
        if not prompt:
            with output_area:
                clear_output()
                print("❌ Please enter a prompt!")
            return
        
        with output_area:
            clear_output()
            print("🤖 Generating response...")
            
            # Generate response
            response = generate_response(
                current_image['tensor'],
                prompt,
                max_new_tokens=max_tokens_slider.value,
                temperature=temperature_slider.value,
                top_p=top_p_slider.value
            )
            
            clear_output()
            display_image_and_response(current_image['pil'], prompt, response)
    
    # Connect event handlers
    sample_dropdown.observe(on_sample_change, names='value')
    file_upload.observe(on_upload_change, names='value')
    prompt_dropdown.observe(on_prompt_change, names='value')
    generate_button.on_click(on_generate_click)
    
    # Layout
    image_section = widgets.VBox([
        widgets.HTML("<h3>📸 Image Input</h3>"),
        file_upload,
        sample_dropdown
    ])
    
    prompt_section = widgets.VBox([
        widgets.HTML("<h3>💬 Prompt Input</h3>"),
        prompt_dropdown,
        prompt_text
    ])
    
    params_section = widgets.VBox([
        widgets.HTML("<h3>⚙️ Generation Parameters</h3>"),
        max_tokens_slider,
        temperature_slider,
        top_p_slider,
        generate_button
    ])
    
    interface = widgets.VBox([
        widgets.HTML("<h2>🔮 VLM Interactive Inference</h2>"),
        widgets.HBox([image_section, prompt_section, params_section]),
        widgets.HTML("<h3>📋 Results</h3>"),
        output_area
    ])
    
    return interface

# Create and display the interface
interface = create_inference_interface()
display(interface)

## Manual Testing (Alternative)

If the interactive widgets don't work in your environment, you can use this cell for manual testing.

In [ ]:
# Manual testing - modify these variables and run the cell
IMAGE_PATH = "path/to/your/image.jpg"  # Update this path
PROMPT = "What do you see in this image?"  # Update this prompt

# Generation parameters
MAX_TOKENS = 256
TEMPERATURE = 0.7
TOP_P = 0.9

# Uncomment and run the following code for manual testing:
"""
try:
    # Process the image
    pixel_values, pil_image = process_image(IMAGE_PATH)
    
    # Generate response
    response = generate_response(
        pixel_values, 
        PROMPT, 
        max_new_tokens=MAX_TOKENS,
        temperature=TEMPERATURE,
        top_p=TOP_P
    )
    
    # Display results
    display_image_and_response(pil_image, PROMPT, response)
    
except Exception as e:
    print(f"Error: {e}")
    print("Please check the IMAGE_PATH and make sure the model is loaded correctly.")
"""

print("Update IMAGE_PATH and PROMPT variables above, then uncomment and run the code block for manual testing.")

## Tips and Troubleshooting

### 🎯 Usage Tips:
1. **Image Upload**: Supports common formats (PNG, JPG, JPEG, BMP, GIF)
2. **Prompts**: Try different types of questions:
   - Descriptive: "What do you see in this image?"
   - Analytical: "What is the mood of this scene?"
   - Specific: "Are there any animals in this image?"
3. **Generation Parameters**:
   - **Temperature**: Lower (0.1-0.5) for more focused responses, higher (0.7-1.0) for more creative responses
   - **Top-p**: Controls diversity; 0.9 is usually good
   - **Max Tokens**: Increase for longer responses

### 🔧 Troubleshooting:
- **Model loading errors**: Check that `MODEL_DIR` points to your trained model
- **CUDA out of memory**: Reduce batch size or use CPU instead
- **Widget not responding**: Try the manual testing section instead
- **Slow inference**: This is normal for larger models; consider using a GPU

### 📊 Model Performance Notes:
- The model's performance depends on the training data and number of training epochs
- For better results, ensure your model was trained on diverse image-text pairs
- The model should work best on image types similar to its training data